#### Early Intervention Simulation
##### In this section, an early support intervention is simulated for students predicted to be at moderate risk of course incompletion (completion probability between 0.2 and 0.5). Students in this range represent a “movable middle” — they are neither certain to complete nor highly unlikely to complete, making them suitable candidates for targeted intervention.

##### We randomly split these students into two groups:
- No-help group
- Help group
##### For the help group, we simulate the effect of reminder emails and course refinement by increasing engagement metrics (active days by 30% and total events by 20%). We then re-run the trained model to estimate the expected change in completion rates and quantify the potential impact of early intervention.

In [39]:
# Importing the required libraries

import pandas as pd
import numpy as np
import sys
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

In [40]:
import src
print("src is now discoverable")

src is now discoverable


In [41]:
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [ ]:
# IMPORTING THE PATHS
BASE_DIR = Path.cwd().parents[0]
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "saved_model"

In [43]:
# Importing the test dataset:

test_df = pd.read_csv("../data/processed/X_test.csv")
test_df.head()

,course_id,viewed,explored,final_cc_cname_DI,LoE_DI,YoB,gender,start_time_DI,last_event_DI,nevents,ndays_act,nchapters,nforum_posts,no_of_courses_registered,no_of_courses_explored
0,HarvardX/CB22x/2013_Spring,1,0,United States,NaN,NaN,NaN,2013-02-18,2013-03-17,70.0,3.0,3.0,0,7,0
1,HarvardX/CS50x/2012,1,0,United States,NaN,NaN,NaN,2012-10-20,NaN,NaN,12.0,3.0,0,7,0
2,HarvardX/ER22x/2013_Spring,1,0,United States,NaN,NaN,NaN,2013-02-23,2013-06-14,17.0,2.0,2.0,0,7,0
3,HarvardX/CS50x/2012,0,0,United States,NaN,NaN,NaN,2012-09-04,NaN,NaN,NaN,NaN,0,1,0
4,HarvardX/CS50x/2012,0,0,United States,NaN,NaN,NaN,2012-09-05,NaN,NaN,NaN,3.0,0,1,0


In [44]:
# Making a copy of the dataset:
X_test = test_df.copy()

In [45]:
# LOAD THE TRANSFORMERS
from src.features.custom_transformers import (
    FeatureEngineer,
    MissingValueImputer,
    MappingEncoder,
    FrequencyEncoder,
    OutlierHandler
)

In [46]:
# LOAD THE MODEL
model_path = MODEL_DIR / "xgb_final_model.joblib"
xgb_best_model = joblib.load(model_path)

In [ ]:
# Getting the prediction probabilities, creating a new column "pred_prob" and assigning the probabilities:
y_pred_proba = xgb_best_model.predict_proba(X_test)[:, 1]
test_df["pred_prob"] = y_pred_proba

In [ ]:
# Selecting the students who are at the risk of incompletion, but still at not hopeless stage, yet not guaranteed
# to complete, and helping half of those students to see if the completion probability increases.
# These students can be those having completion probabilities between 0.2 and 0.5
at_risk_df = test_df[
    (test_df["pred_prob"] >= 0.2) &
    (test_df["pred_prob"] <= 0.5)
].copy()

In [49]:
at_risk_df.head()

,course_id,viewed,explored,final_cc_cname_DI,LoE_DI,YoB,gender,start_time_DI,last_event_DI,nevents,ndays_act,nchapters,nforum_posts,no_of_courses_registered,no_of_courses_explored,pred_prob
95,HarvardX/CS50x/2012,1,1,India,NaN,NaN,NaN,2012-12-18,2013-07-11,91.0,18.0,12.0,0,4,2,0.493564
104,HarvardX/ER22x/2013_Spring,1,0,United States,NaN,NaN,NaN,2013-01-09,2013-07-06,495.0,16.0,16.0,0,1,0,0.238509
248,HarvardX/ER22x/2013_Spring,1,0,Canada,NaN,NaN,NaN,2012-12-22,2013-07-28,368.0,10.0,10.0,0,5,1,0.432014
612,HarvardX/CS50x/2012,1,1,United States,NaN,NaN,NaN,2012-09-12,2013-08-01,306.0,24.0,12.0,0,4,1,0.373157
733,HarvardX/ER22x/2013_Spring,1,1,Other Middle East/Central Asia,NaN,NaN,NaN,2013-01-23,2013-06-02,665.0,2.0,32.0,0,5,1,0.209385


In [ ]:
# Now, we randomly assign groups to these students. Half of the students may receive help, i.e., get
# remainder emails and refined course materials, and the other half does not. So, we assign groups "help" and "no_help" to at_risk_df:
np.random.seed(42)

at_risk_df["group"] = np.random.choice(
    ["no_help", "help"],
    size=len(at_risk_df)
)

In [51]:
# Checking the distribution of the "help" and "no_help" categories:
at_risk_df["group"].value_counts()

group
help       521
no_help    504
Name: count, dtype: int64

In [52]:
# Almost half of the students has received help and half does not.

In [53]:
# Now we assume that for those students who received help, the number of active days increase by 30%,
# and number of events (nevents) increase by 20%.
# We simulate this activity increase only for the "help" group:

at_risk_df.loc[
    at_risk_df["group"] == "help",
    "ndays_act"
] *= 1.30

at_risk_df.loc[
    at_risk_df["group"] == "help",
    "nevents"
] *= 1.20

In [54]:
# Rounding the ndays_act column:
at_risk_df["ndays_act"] = at_risk_df["ndays_act"].round()

In [55]:
# Dropping the columns "pred_prob" and "group" and assigning it to a new variable:
X_modified = at_risk_df.drop(columns=["pred_prob", "group"])

# Rerunning the model on the new dataset (i.e., the dataset in which we increased ndays_act) to get the
# new probabilities
new_probs = xgb_best_model.predict_proba(X_modified)[:, 1]

# Assigning the new probabilities to the at_risk_df:
at_risk_df["new_pred_prob"] = new_probs

In [56]:
# Now we calculate the average predicted probability for each group and compare it to see if the 
# completion probability has increased.

no_help_rate = at_risk_df[
    at_risk_df["group"] == "no_help"
]["new_pred_prob"].mean()

help_rate = at_risk_df[
    at_risk_df["group"] == "help"
]["new_pred_prob"].mean()

lift = help_rate - no_help_rate

In [57]:
print(f"No Help Completion Probability: {no_help_rate:.4f}")
print(f"Help Completion Probability: {help_rate:.4f}")
print(f"Estimated Lift: {lift:.4f}")

No Help Completion Probability: 0.3287
Help Completion Probability: 0.3996
Estimated Lift: 0.0710


#### The early intervention simulation increased the completion rate by 7.1%.